# Module 8 Assignment: Give Your Persona Agent Memory With Sessions

Lab 7 added a `SQLiteSession` to Study Buddy: first in memory, to fix two `Runner.run` calls that
did not remember each other, then file backed, so the conversation survives a kernel restart. The
Gradio `chat()` function got simpler in the process, no more rebuilding `history` by hand.

This assignment makes you do the same thing to your **own** Persona Agent, the one you rebuilt on
the OpenAI Agents SDK in the Module 7 assignment. Nothing about your agent's persona, notes, or
three tools needs to change here: you are giving it memory, not redesigning it.

**What you'll build:**
1. Your Module 7 assignment's Agent, Azure configuration, and three tools, carried over unchanged
2. A quick demonstration that two separate `Runner.run` calls do not remember each other
3. A file backed `SQLiteSession` that fixes it
4. A simplified Gradio `chat()` function that uses `session=session` instead of rebuilding `history`

**Provider:** Azure OpenAI via APIM, deployment `gpt-5.4-ptu`, same as every lab and assignment so
far.

If you get stuck on any step, look at how Lab 7 made the same kind of change to Study Buddy, and do
the analogous thing to your own agent.

## 0. Install the packages (run once)

Same as Lab 7: no new package beyond what your Module 7 assignment already installed.
`SQLiteSession` ships with `openai-agents` itself.

In [1]:
!uv pip install --upgrade openai openai-agents pypdf python-dotenv requests gradio

Resolved 79 packages in 8.41s
 Downloaded openai
Prepared 9 packages in 26.78s
Uninstalled 9 packages in 5.28s
Installed 9 packages in 5.95s
 - certifi==2026.7.22
 + certifi==2026.10.19
 - fastapi==0.143.2
 + fastapi==0.156.2
 - filelock==4.2.0
 + filelock==4.2.4
 - griffelib==2.3.2
 + griffelib==2.4.0
 - httpcore2==2.13.1
 + httpcore2==2.15.1
 - httpx2==2.13.1
 + httpx2==2.15.1
 - openai==3.31.0
 + openai==3.35.1
 - pypdf==6.20.1
 + pypdf==6.21.1
 - starlette==1.8.0
 + starlette==1.9.0


## 1. Imports

**TODO:** Copy the imports from your Module 7 assignment, plus one new name: `SQLiteSession`,
imported from `agents` alongside `Agent` and `Runner`, the same way Lab 7, Step 1 added it.

In [3]:
from dotenv import load_dotenv
import os
import requests
from openai import AsyncAzureOpenAI
from agents import (
    Agent,
    Runner,
    function_tool,
    OpenAIChatCompletionsModel,
    set_tracing_disabled,
    SQLiteSession,
)
from pypdf import PdfReader
import gradio as gr


c:\Users\colby\Desktop\IT3025C Labs\Lab 7\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Load your Azure OpenAI credentials from `.env`

Identical to the Module 7 assignment. Sessions do not change which provider you talk to.

In [4]:
from dotenv import load_dotenv
import os

load_dotenv(override=True)

AZURE_OPENAI_ENDPOINT = os.getenv("AZURE_OPENAI_ENDPOINT")
AZURE_OPENAI_API_KEY = os.getenv("AZURE_OPENAI_API_KEY")
DEPLOYMENT = "gpt-5.4-ptu"

print("Endpoint loaded:", bool(AZURE_OPENAI_ENDPOINT))
print("API key loaded:", bool(AZURE_OPENAI_API_KEY))

Endpoint loaded: True
API key loaded: True


## 3. Configure the OpenAI Agents SDK for Azure OpenAI

**TODO:** Copy this setup from your Module 7 assignment, unchanged: `set_tracing_disabled(True)`,
your `AsyncAzureOpenAI` client, and `OpenAIChatCompletionsModel`. A session is a feature of
`Runner.run`, not of the model or the client, so nothing here needs to change.

In [5]:
set_tracing_disabled(True)

azure_client = AsyncAzureOpenAI(
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
    api_key=AZURE_OPENAI_API_KEY,
    api_version="2024-10-21",
)

azure_model = OpenAIChatCompletionsModel(
    model=DEPLOYMENT,
    openai_client=azure_client,
)

print("Azure model configured for the OpenAI Agents SDK.")

Azure model configured for the OpenAI Agents SDK.


## 4. Your persona, notes, and three tools

**TODO:** Bring over your persona, your notes, and all three of your `@function_tool` decorated
tools from the Module 7 assignment, unchanged. None of them need to change for this assignment: a
session only affects how conversation history is stored and replayed, not what a tool does.

In [6]:
persona = (
    "Finance Coach, a calm and practical study assistant who helps students review liquidity, "
    "working capital, cash management, and breakeven analysis. Finance Coach asks focused "
    "questions, explains formulas clearly, and helps students identify topics they should review again."
)

reader = PdfReader("Notes/module5_notes.pdf")
notes = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        notes += text

print(persona)
print(notes[:300], "...")

Finance Coach, a calm and practical study assistant who helps students review liquidity, working capital, cash management, and breakeven analysis. Finance Coach asks focused questions, explains formulas clearly, and helps students identify topics they should review again.
Module 5 Study Notes: Liquidity, Working Capital, and Breakeven
This study sheet summarizes Module 06: Liquidity, Working Capital, and Breakeven. The
main ideas are how entrepreneurs measure short-term financial health, manage cash and
working capital, and calculate the revenue level needed for surv ...


In [7]:
@function_tool
def my_tool(value: str) -> str:
    """Save a finance question or topic that the student needs to review.

    Args:
        value: The question or topic that needs to be reviewed.
    """
    print(f"Tool called to log review item: {value}")
    with open("review_items.txt", "a", encoding="utf-8") as f:
        f.write(value + "\n")
    return "Review item logged successfully."

print("Tool ready:", my_tool.name)

Tool ready: my_tool


In [8]:
@function_tool
def api_tool(currency: str) -> str:
    """Check the exchange rate from US dollars to another currency.

    Args:
        currency: A currency code such as EUR, GBP, or JPY.
    """
    print(f"Tool called to check the exchange rate for {currency}")
    currency = currency.upper()

    try:
        response = requests.get(
            f"https://api.frankfurter.dev/v2/rate/USD/{currency}",
            timeout=20
        )
        if response.status_code != 200:
            return "I could not find that exchange rate."
        data = response.json()
    except requests.exceptions.RequestException:
        return "The exchange rate website is not available right now."
    except ValueError:
        return "The exchange rate website returned invalid information."

    if "rate" not in data:
        return "I could not find an exchange rate for that currency."

    rate = data["rate"]
    return f"1 US dollar is about {rate} {currency}."

print("Tool ready:", api_tool.name)

Tool ready: api_tool


In [9]:
@function_tool
def third_tool() -> str:
    """Check how many items the student has saved for review."""
    print("Tool called to check review progress")

    try:
        with open("review_items.txt", "r", encoding="utf-8") as file:
            review_items = file.readlines()
    except FileNotFoundError:
        return "You do not have any review items yet."

    count = len(review_items)
    return f"You have {count} item(s) to review."

print("Tool ready:", third_tool.name)

Tool ready: third_tool


## 5. Build your Agent

**TODO:** Copy this from your Module 7 assignment, unchanged: `name`, `instructions`, your `tools`
list, and `model=azure_model`.

In [10]:
instructions = f"""
# Your role

You are {persona}

# Context

Use these notes to quiz the student:

{notes}

# Rules

Ask only one question at a time, and only ask quiz questions based on the notes.
Wait for the student to answer before asking another question.
Tell the student if their answer is correct or incorrect and give a short explanation.

If the student gets a question wrong or says "I don't know", use my_tool to save the
question for review before moving on.

If the student asks for an exchange rate, use api_tool to find the exchange rate.

If the student asks how many review items they have, use third_tool to check their progress.

Keep your answers calm, simple, and encouraging.
"""

finance_coach = Agent(
    name="Finance Coach",
    instructions=instructions,
    tools=[my_tool, api_tool, third_tool],
    model=azure_model,
)

print("Agent built:", finance_coach.name)

Agent built: Finance Coach


## 6. Confirm the problem: two calls that do not know each other

**TODO:** Run your agent twice in a row with `await Runner.run(your_agent, "...")`, the same way
Lab 7, Step 7 did with Study Buddy. Pick a first message that gives the agent some information (for
example, ask it a question whose answer it needs to remember), then a second message that only
makes sense if the agent remembers the first (for example, "what did I just ask you?" or "say that
again"). Confirm the second reply shows the agent has no idea what you mean.

In [11]:
result = await Runner.run(
    finance_coach,
    "Remember that I prefer to study breakeven analysis."
)
print(result.final_output)

Got it — I’ll emphasize breakeven analysis in our quiz while still staying within the module notes.

First question:

What is the formula for contribution margin in terms of the variable cost revenue ratio (VCRR)?


In [12]:
result = await Runner.run(
    finance_coach,
    "Which topic did I say I prefer to study?"
)
print(result.final_output)

You haven’t told me a preferred topic in this chat yet.

If you want, I can quiz you on one of these:
- liquidity ratios
- working capital management
- cash conversion cycle
- receivables, inventory, and payables
- breakeven analysis

Which topic do you prefer?


## 7. Fix it with a file backed `SQLiteSession`

**TODO:** Create a `SQLiteSession`, the same way Lab 7, Step 9 did: give it a `session_id` string,
and a `db_path` so the conversation is written to a file and survives a kernel restart. Then repeat
the same two messages from Step 6, this time passing `session=your_session` to both calls, and
confirm the second reply now correctly remembers the first.

In [13]:
your_session = SQLiteSession(
    "finance_coach_assignment",
    "finance_coach_assignment.db"
)

In [14]:
result = await Runner.run(
    finance_coach,
    "Remember that I prefer to study breakeven analysis.",
    session=your_session,
)
print(result.final_output)

Got it — I’ll emphasize breakeven analysis in the quiz.

First question: What is the formula for contribution margin in terms of VCRR?


In [15]:
result = await Runner.run(
    finance_coach,
    "Which topic did I say I prefer to study?",
    session=your_session,
)
print(result.final_output)


You said you prefer to study breakeven analysis.

Now a quick question: What does VCRR stand for?


## 8. Give it a real chat UI with Gradio

**TODO:** Copy this skeleton from Lab 7, Step 11 exactly, structure and all. Notice what is
different from your Module 7 assignment's version: `history` is no longer rebuilt inside `chat()`
at all, `session=your_session` already gives `Runner.run` everything it needs to remember the
conversation.

Try a conversation that would trigger more than one of your three tools across a few turns, then
interrupt the cell (stop button) when you are done chatting.

In [16]:
async def chat(message, history):
    result = await Runner.run(
        finance_coach,
        message,
        session=your_session
    )
    return result.final_output

gr.ChatInterface(chat).launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


Tool called to log review item: Review question: What is the formula for survival revenues at EBDAT breakeven? Answer: Survival Revenues = Cash Fixed Costs / (1 - VCRR).
Tool called to check the exchange rate for JPY
Tool called to check review progress


## Reflection Questions

Answer these in a new Markdown cell at the end of your notebook:

1. In Step 6, what exactly did your agent get wrong on the second call, and why? Point to the
   specific piece of information it had no way of knowing.

## 1 Response: 
- Step 6 was about telling the agent that I prefered to talk about breakeven analysis. Afterwards, it didn't reliably know it, because it had no session containing the previous message. It said "You haven’t told me a preferred topic in this chat yet.

If you want, I can quiz you on one of these:
- liquidity ratios
- working capital management
- cash conversion cycle
- receivables, inventory, and payables
- breakeven analysis

Which topic do you prefer?"

2. Compare your Gradio `chat()` function here to the one in your Module 7 assignment. What code did
   you get to delete, and what does `session=your_session` do in its place?

## 2 Response 
- In Module 7, the chat() copiede Gradio's histroy and put it into a list and had the last user message before calling Runner.run. Now the session=your_session retrieves the conversation and saves it each time. 

3. Your `your_session` was created with a `db_path`. If you had left `db_path` out entirely, what
   would still work the same, and what would break the next time you restarted the kernel?

## 3 Response
-Without db_path, the session would remember each turn as the Python process ran, but the negative is that the history was lost each time the kernel got reset. The database saves history to a file that can be reconnected each time. 

4. `SQLiteSession` also has a `pop_item()` method, demonstrated in Lab 7, Step 10, for undoing the
   last turn. Describe one realistic situation in your own agent's domain where you would want to
   call it.

## 4 Response 
- If the Finance Coach misunderstood a question and I wanted to have it not saved I could you pop_item(). to remove the last conversation before trying again

## Submission Checklist

- Remove your `.env` file before submitting; it contains your API key.
- Remove any `.db` file your session created; it may contain real conversation content.
- Run the notebook top to bottom and confirm it executes cleanly from a fresh kernel.
- Confirm Step 6 shows the agent failing to remember, and Step 7 shows the same scenario working
  correctly once `session=your_session` is added.
- Answer all four reflection questions in a Markdown cell at the end of the notebook.
- Rename the file to include your name, for example `Module8Assignment_JaneDoe.ipynb`, then submit
  it.